In [2]:
import json
import re
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import accelerate

c:\Users\刘心怡\Desktop\大模型春招\llms\project\qwen-math-rl\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [72]:
# ==========================
# 1. 加载 GSM8K
# ==========================
dataset = load_dataset("gsm8k", "main")
test_data = dataset["test"].select(range(20))

In [ ]:
查看数据集结构

In [81]:
# 查看数据集的结构
print(dataset)

# 获取测试集的前 5 条数据
test_data = dataset["test"].select(range(5))

# 打印前 5 条测试集样本
for idx, sample in enumerate(test_data):
    print(f"Sample {idx + 1}:")
    print(f"Question: {sample['question']}")
    print(f"Answer: {sample['answer']}")
    print("=" * 60)

DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 7473
    })
    test: Dataset({
        features: ['question', 'answer'],
        num_rows: 1319
    })
})
Sample 1:
Question: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?
Answer: Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.
She makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.
#### 18
Sample 2:
Question: A robe takes 2 bolts of blue fiber and half that much white fiber.  How many bolts in total does it take?
Answer: It takes 2/2=<<2/2=1>>1 bolt of white fiber
So the total amount of fabric is 2+1=<<2+1=3>>3 bolts of fabric
#### 3
Sample 3:
Question: Josh decides to try flipping a house.  He buys a house for $80,000 and then puts in $50,00

In [80]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen3-0.6B"

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

c:\Users\刘心怡\Desktop\大模型春招\llms\project\qwen-math-rl\.venv\lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in D:\hf_cache\hub\models--Qwen--Qwen3-0.6B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 311/311 [00:00<00:00, 360.72it/s, Materializing param=model.norm.weight]          

In [ ]:
# ==========================
# 2. 加载模型
# ==========================
model_name = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,
    device_map="auto"
)

OSError: Qwen3-0.6B is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`

In [73]:
import re, json
from tqdm.auto import tqdm
from transformers import StoppingCriteria, StoppingCriteriaList

class StopOnHashAnswer(StoppingCriteria):
    def __init__(self, tokenizer, window=256):
        self.tokenizer = tokenizer
        self.window = window
        # 只要出现一行 #### 数字（允许句号），就停止
        self.pattern = re.compile(
            r"(?:####\s*[-+]?\d[\d,]*(?:\.\d+)?\s*(?:[.。])?\s*$|\\boxed\{\s*[-+]?\d[\d,]*(?:\.\d+)?\s*\}\s*$)"
            )

    def __call__(self, input_ids, scores, **kwargs):
        tail = input_ids[0][-self.window:]
        text = self.tokenizer.decode(tail, skip_special_tokens=False)
        return self.pattern.search(text) is not None

stopping_criteria = StoppingCriteriaList([StopOnHashAnswer(tokenizer)])

def extract_model_answer(text: str):
    # 0) boxed
    m = re.search(r"\\boxed\{\s*([-+]?\d[\d,]*(?:\.\d+)?)\s*\}", text)
    if m:
        return m.group(1).replace(",", "")
    # 1) ####
    m = re.search(r"####\s*([-+]?\d[\d,]*(?:\.\d+)?)", text)
    if m:
        return m.group(1).replace(",", "")
    # 2) Final Answer:
    m = re.search(r"(?:Final\s*Answer|Answer)\s*[:=]\s*([-+]?\d[\d,]*(?:\.\d+)?)", text, re.I)
    if m:
        return m.group(1).replace(",", "")
    # 3) 去掉步骤编号兜底
    cleaned = re.sub(r"(?m)^\s*\d+\.\s+", "", text)
    nums = re.findall(r"[-+]?\d[\d,]*(?:\.\d+)?", cleaned)
    return nums[-1].replace(",", "") if nums else None

# system 简短即可
system_prompt = "You are a math assistant."

output_file = "predictions.jsonl"

with open(output_file, "w", encoding="utf-8") as f:
    for sample in tqdm(test_data, total=len(test_data)):
        question = sample["question"]
        gt_answer = extract_answer(sample["answer"])

        # 把格式要求放在 user（更有效），并尽量减少“模板复读”
        user_prompt = (
            question
            + "\n\nGive the final answer at the end in ONE of these formats:\n"
              "1) #### <number>\n"
              "OR 2) \\boxed{<number>}\n"
              "Do not write anything after the final answer line."
        )

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]

        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt").to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=512,
                do_sample=False,
                temperature=0.0,
                pad_token_id=tokenizer.eos_token_id,
                stopping_criteria=stopping_criteria,
            )

        gen_ids = outputs[0][inputs["input_ids"].shape[-1]:]
        model_output = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

        pred_answer = extract_model_answer(model_output)

        f.write(json.dumps({
            "question": question,
            "model_output": model_output,
            "pred_answer": pred_answer,
            "gt_answer": gt_answer
        }, ensure_ascii=False) + "\n")

print("predictions.jsonl 生成完成")

  0%|          | 0/20 [00:00<?, ?it/s]

100%|██████████| 20/20 [21:49<00:00, 65.48s/it]

predictions.jsonl 生成完成


In [74]:
# 1) 看看是不是有 ####（如果你要求了的话）
print("has ####:", "####" in model_output)

# 2) 看生成是否因为 max_new_tokens 截断
print("gen_len:", len(gen_ids), "max_new_tokens:", 512)
print("tail:", model_output[-200:])

has ####: False
gen_len: 325 max_new_tokens: 512
tail: t{distance}}{\text{time}} = \frac{8 \text{ miles}}{2 \text{ hours}} = 4 \) miles per hour.

Therefore, Marissa needs to walk the remaining distance at an average speed of \(\boxed{4}\) miles per hour.


暂时跳过has ####: False（没有###的问题）

In [75]:
# sanity check
import json
with open("predictions.jsonl","r",encoding="utf-8") as f:
    for i,line in enumerate(f):
        if i==3: break
        d=json.loads(line)
        print("Pred:", d["pred_answer"], "GT:", d["gt_answer"])
        print("Output:", d["model_output"][:200], "...\n")

Pred: 18 GT: 18
Output: To determine how much Janet makes at the farmers' market each day, we need to follow these steps:

1. Calculate the total number of eggs laid by the ducks in one day.
2. Determine how many eggs are so ...

Pred: 3 GT: 3
Output: To determine the total number of bolts needed for the robe, we need to calculate the amount of each type of fiber required and then sum them up.

1. **Blue Fiber:**
   - The robe requires 2 bolts of b ...

Pred: 70000 GT: 70000
Output: To determine Josh's profit from flipping his house, we need to follow these steps:

1. **Calculate the total cost of the house:**
   - Cost of the house: $80,000
   - Cost of repairs: $50,000
   - Tot ...



#### 分类错误情形

In [76]:
import json, re
from collections import Counter, defaultdict

path = "predictions.jsonl"
wrong = []
stats = Counter()
buckets = defaultdict(list)

def is_truncated(text: str):
    # 非严格，但够用：以省略号/未完句/最后没有数字迹象为信号
    tail = text.strip()[-50:]
    return ("..." in tail) or (tail.endswith(",")) or (tail.endswith(":"))

def has_boxed(text: str):
    return r"\boxed" in text

def has_hash(text: str):
    return "####" in text

with open(path, encoding="utf-8") as f:
    for line in f:
        ex = json.loads(line)
        pred = ex.get("pred_answer")
        gt = ex.get("gt_answer")
        out = ex.get("model_output","")
        q = ex.get("question","")

        if pred is None:
            stats["extract_fail"] += 1
            buckets["extract_fail"].append(ex)
            continue

        if str(pred) == str(gt):
            stats["correct"] += 1
            continue

        stats["wrong"] += 1

        # 简单规则打桶
        if is_truncated(out):
            tag = "likely_truncated"
        elif has_boxed(out):
            tag = "boxed_but_wrong"
        elif has_hash(out):
            tag = "hash_but_wrong"
        else:
            tag = "wrong_other"

        buckets[tag].append(ex)

print("Stats:", stats)
for k,v in buckets.items():
    print(k, len(v))

# 导出错题（方便你肉眼看）
def dump(bucket_name, n=20):
    arr = buckets[bucket_name][:n]
    for i,ex in enumerate(arr):
        print("="*60, bucket_name, i)
        print("GT:", ex["gt_answer"], "Pred:", ex["pred_answer"])
        print("Q:", ex["question"][:200])
        print("Out tail:", ex["model_output"][-300:])

# 例：先看看最可能截断的
dump("likely_truncated", n=10)

Stats: Counter({'wrong': 11, 'correct': 9})
boxed_but_wrong 10
wrong_other 1


针对boxed_but_wrong的问题

In [78]:
# 只看前10条 boxed_but_wrong
for i, ex in enumerate(buckets["boxed_but_wrong"][:10]):
    print("=" * 80)
    print(f"[{i}] GT: {ex['gt_answer']} | Pred: {ex['pred_answer']}")
    print("Q:", ex["question"])
    print("\nOUT (tail):\n", ex["model_output"][-500:])

[0] GT: 540 | Pred: 4200
Q: James decides to run 3 sprints 3 times a week.  He runs 60 meters each sprint.  How many total meters does he run a week?

OUT (tail):
  60 meters and runs 3 times a week, the distance for one sprint is:
\[ 60 \text{ meters/sprint} \times 3 \text{ sprints/week} = 180 \text{ meters/week} \]

Next, we calculate the total distance he runs in a week. Since he runs 3 sprints per day and runs 180 meters each sprint, the total distance for one week is:
\[ 180 \text{ meters/sprint} \times 3 \text{ sprints/day} \times 7 \text{ days/week} = 4200 \text{ meters/week} \]

Therefore, the total distance James runs in a week is:
\[ \boxed{4200}
[1] GT: 20 | Pred: 100
Q: Every day, Wendi feeds each of her chickens three cups of mixed chicken feed, containing seeds, mealworms and vegetables to help keep them healthy.  She gives the chickens their feed in three separate meals. In the morning, she gives her flock of chickens 15 cups of feed.  In the afternoon, she gives her chi

第一道题的错误原因：【推理错误】unit_confusion / invented_frequency（单位混乱 + 自己补设定）
第二题：【推理错误】direction_error / subtract_vs_add（加减方向错误）

GPT给的建议
1.更详细严谨的user_prompt限制：
Rules:
- Do NOT introduce any new time units (e.g., days/week) unless explicitly stated.
- Track units for every quantity (per sprint / per session / per week).
- If the question asks for "final/remaining", compute: total_needed - already_given.
- End with: #### <number>

2.评测时更稳：只输出答案，不输出推理（能明显减少这类“话越多越容易编”的错误）。